# Feature Engineering & Location Analysis

## Research Question
> Are property prices in Mexico more influenced by property size or by location?

## What This Notebook Does
This is the final notebook in the analysis pipeline. We engineer a new 
feature — price per m² — to fairly compare properties of different sizes 
and use it to definitively answer our research question:

- Create and analyze the `price_per_m2` feature
- Compare price per m² across states (top 3 vs bottom 3)
- Quantify the location vs size effect using variance decomposition
- Compare price per m² across property types
- Draw a final conclusion to the research question

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load clean dataset
df_raw = pd.read_csv("../data/mexico-real-estate-combined-clean.csv")
print(f"Shape: {df_raw.shape}")
df_raw.head()

## 2. Feature Engineering — Price Per m²

Raw price comparisons are unfair — a large cheap house can cost the same 
as a small expensive apartment. Price per m² normalizes for size, making 
comparisons across properties, states, and types much more meaningful.

**High price_per_m²** → desirable urban location, luxury finishes, scarce land  
**Low price_per_m²** → rural/suburban location, basic finishes, abundant land

In [ ]:
# Create new column 'price_per_m2'
df = (
    df_raw
    .assign(price_per_m2=lambda x: x["price_usd"] / x["area_m2"])
)

print("Price per m² Statistics:")
print(df["price_per_m2"].describe().round(2))
df.head()

## 3. Distribution of Price Per m²

In [ ]:
fig, ax = plt.subplots()
ax.hist(df["price_per_m2"], bins=30, edgecolor="white")
ax.set_title("Distribution of Price per Square Meter")
ax.set_xlabel("Price per m² [USD]")
ax.set_ylabel("Frequency")

# Add median line
median_price_per_m2 = df["price_per_m2"].median()
ax.axvline(
    median_price_per_m2, 
    color="red", 
    linestyle="--",
    label=f"Median: ${median_price_per_m2:.2f}/m²"
)
ax.legend()
plt.tight_layout()
fig.savefig("../images/11_price_per_m2_distribution.png", 
            bbox_inches="tight", dpi=150)
plt.show()

**Finding:** Price per m² is still right-skewed even after normalizing 
for size. The location effect creates the right tail — luxury urban 
apartments in Mexico City can reach $2,000–$3,000/m² while rural 
properties sit around $300–$500/m². The median (red line) is the 
most honest measure of a typical property's value density.

## 4. Price Per m² by State

Which states are the most and least expensive per square meter?
We use the grouped analysis workflow:
1. **Aggregate** — compute statistics per state
2. **Rank** — sort to find extremes  
3. **Visualize** — boxplot top 3 vs bottom 3
4. **Quantify** — variance decomposition

In [ ]:
# Calculate statistics by state
state_price_stats = (
    df
    .groupby("state")["price_per_m2"]
    .agg(["mean", "median", "std"])
    .sort_values(by="median", ascending=False)
    .round(2)
)

print("Price per m² Statistics by State:")
state_price_stats

**Finding:** There is enormous variation across states. The top states 
(Distrito Federal, coastal/tourist states) have median prices multiple 
times higher than the bottom states (rural, agricultural states). 
The `std` column shows that expensive states also have more internal 
inequality — a wide range from modest to luxury within the same state.

## 5. Top 3 vs Bottom 3 States — The Location Effect

In [ ]:
# Top 3 and bottom 3 states by median price per m²
top_states = state_price_stats.head(3).index.tolist()
bottom_states = state_price_stats.tail(3).index.tolist()
target_states = top_states + bottom_states

print("Top 3 states:   ", top_states)
print("Bottom 3 states:", bottom_states)

# Filter and prepare data
df_states = df[df["state"].isin(target_states)]
data_to_plot = [
    df_states[df_states["state"] == state]["price_per_m2"]
    for state in target_states
]

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(data_to_plot, tick_labels=target_states, vert=False)
ax.set_title("Price per m² — Most vs Least Expensive States")
ax.set_xlabel("Price per m² [USD]")
ax.set_ylabel("State")
plt.tight_layout()
fig.savefig("../images/12_top_vs_bottom_states.png", 
            bbox_inches="tight", dpi=150)
plt.show()

**Finding:** The gap between the top and bottom states is enormous. 
The median price per m² of Distrito Federal likely exceeds or approaches 
the maximum of the least expensive states. This is the **Location Effect** 
in its purest form — where a property is located matters more than 
almost any other factor.

## 6. Quantifying Location vs Size — Variance Decomposition

Boxplots show the location effect visually. Now we quantify it:
what fraction of total price variation comes from between-state 
differences vs within-state differences?

- **Between-state variance** → how much states differ from each other
- **Within-state variance** → how much prices vary within the same state

If between > within → **location explains more than size**

In [ ]:
# Size-price correlation
size_price_correlation = df[["area_m2", "price_usd"]].corr().iloc[0, 1]
print(f"Correlation between area and price: {size_price_correlation:.3f}")

# Variance decomposition
total_variance = df["price_usd"].var()
state_means = df.groupby("state")["price_usd"].mean()
between_variance = state_means.var()
within_variance = df.groupby("state")["price_usd"].var().mean()

between_pct = (between_variance / total_variance) * 100
within_pct = (within_variance / total_variance) * 100

print(f"\nVariance Decomposition:")
print(f"  Between-state variance : {between_pct:.1f}%")
print(f"  Within-state variance  : {within_pct:.1f}%")
print(f"  Ratio (Between/Within) : {between_variance/within_variance:.2f}")

**Finding:** The variance decomposition gives us a definitive answer.
A ratio above 1.0 means between-state differences explain more price 
variation than within-state differences — **location dominates**.

Combined with the correlation of r = 0.59 between size and price, 
the evidence is clear: knowing which state a property is in tells 
you more about its price than knowing its size.

## 7. Price Per m² by Property Type

In Lesson 2, houses and apartments had similar total prices. 
But are they equally valuable per square meter?

In [ ]:
# Summary statistics by property type
property_type_stats = (
    df
    .groupby("property_type")["price_per_m2"]
    .agg(["mean", "median", "count"])
    .round(2)
)
print("Price per m² by Property Type:")
print(property_type_stats)

# Boxplot
prop_types = df["property_type"].unique()
data_by_type = [
    df[df["property_type"] == p]["price_per_m2"] 
    for p in prop_types
]

fig, ax = plt.subplots()
ax.boxplot(data_by_type, tick_labels=prop_types, vert=False)
ax.set_title("Price per m² by Property Type")
ax.set_xlabel("Price per m² [USD]")
ax.set_ylabel("Property Type")
plt.tight_layout()
fig.savefig("../images/13_price_per_m2_by_type.png", 
            bbox_inches="tight", dpi=150)
plt.show()

**Finding:** Apartments have a significantly higher median price per m² 
than houses. This reveals what raw price comparisons hid in Lesson 2 — 
apartments are actually more expensive per unit of space because they 
are concentrated in urban centers where land is scarce.

This confirms the finding from Lesson 3: size explains less of apartment 
prices (r ≈ 0.52) because location and amenities drive apartment value 
more than raw square footage.

## 8. Combined Analysis — State and Property Type

In [ ]:
# Group by both state and property type
combined_stats = (
    df
    .groupby(["state", "property_type"])["price_per_m2"]
    .agg(["median", "count"])
    .round(2)
)

# Filter for groups with at least 5 properties
combined_stats_filtered = combined_stats[combined_stats["count"] >= 5]
combined_stats_filtered = combined_stats_filtered.sort_values(
    by="median", ascending=False
)

print("Top 15 State-Property Type Combinations:")
print(combined_stats_filtered.head(15))
print("\nBottom 10 State-Property Type Combinations:")
print(combined_stats_filtered.tail(10))

## 9. Final Conclusion

### Answer to the Research Question
> *"Are property prices in Mexico more influenced by property size or location?"*

**Location is the stronger driver of property prices in Mexico.**

The evidence across all four lessons consistently points to this conclusion:

| Evidence | Finding |
|---|---|
| Boxplot by state (L2) | Dramatic price differences across states |
| Small multiples (L2) | Size-price relationship changes completely by state |
| Correlation by state (L3) | National r=0.59 hides state values from ~0.0 to 0.9+ |
| Variance decomposition (L4) | Between-state variation exceeds within-state variation |
| Price per m² by state (L4) | Top states cost multiple times more than bottom states |

### Key Takeaway
> Knowing which state a property is in tells you more about its 
> price per m² than knowing how large it is. Location is not just 
> one factor among many — it is the dominant driver of property 
> value in the Mexican real estate market.

### Limitations & Next Steps
- State-level analysis is broad — neighbourhood-level data would 
  reveal even stronger location effects
- The dataset captures one time period — temporal factors like 
  seasonality and economic cycles are not captured
- Additional features like year built, condition, and proximity 
  to transit would strengthen the model